## 시나리오

사내 문서 100개에서 회사명이 바뀌었을 때, 또는 오타가 발견되었을 때 일일이 열어서 바꾸는 대신 파이썬 한 번으로 처리합니다.

1. 지정된 폴더에서 `.hwp` 파일 전부 찾기
2. 각 파일을 열고
3. `replace_all` 로 치환
4. 원본 또는 출력 폴더에 저장
5. 처리 결과 요약 출력

## 전체 코드

```python
from pathlib import Path
from hwpapi.core import App


def bulk_replace(
    input_dir: str,
    replacements: dict,       # {"옛말": "새말", ...}
    output_dir: str = None,   # None이면 원본에 덮어쓰기
    pattern: str = "*.hwp",
    recursive: bool = True,
) -> dict:
    """
    폴더 안의 HWP 파일들을 일괄 치환합니다.

    Returns:
        {'processed': N, 'total_replacements': M, 'files': [...]}
    """
    in_root = Path(input_dir)
    out_root = Path(output_dir) if output_dir else in_root
    out_root.mkdir(parents=True, exist_ok=True)

    files = list(in_root.rglob(pattern) if recursive else in_root.glob(pattern))
    print(f"대상 파일 {len(files)}개 발견")

    app = App(is_visible=False)   # 눈에 안 보이게 처리하면 훨씬 빠름
    results = []
    total_replacements = 0

    for i, path in enumerate(files, 1):
        print(f"[{i}/{len(files)}] {path.name} 처리 중...")
        app.open(str(path))

        file_count = 0
        for old, new in replacements.items():
            count = app.replace_all(old, new) or 0
            file_count += count
            print(f"  '{old}' → '{new}': {count}회")

        # 출력 경로 결정
        rel = path.relative_to(in_root)
        out_path = out_root / rel
        out_path.parent.mkdir(parents=True, exist_ok=True)

        app.save(str(out_path))
        app.close()

        results.append({"file": str(path), "replacements": file_count})
        total_replacements += file_count

    return {
        "processed": len(files),
        "total_replacements": total_replacements,
        "files": results,
    }


# 사용 예
stats = bulk_replace(
    input_dir=r"C:\docs\2026",
    replacements={
        "(주)이전회사명": "(주)새회사명",
        "Old Address": "New Address",
        "ceo@old.com": "ceo@new.com",
    },
    output_dir=r"C:\docs\2026_updated",  # 원본 보존
)

print(f"\n✅ 완료: {stats['processed']}개 파일, {stats['total_replacements']}회 치환")
```

## 핵심 포인트

- **`is_visible=False`** 로 앱을 띄우면 화면 갱신이 없어 훨씬 빠릅니다. 수십~수백 파일 처리 시 권장.
- **원본 보존**: `output_dir` 을 따로 지정하면 원본 파일은 그대로 두고 치환본만 새 폴더에 저장합니다. 중간에 문제가 생겨도 원본은 안전합니다.
- **`rglob`** 으로 하위 폴더까지 재귀 검색할 수 있습니다.
- **`replace_all` 의 반환값** 은 치환 횟수입니다. 파일별 통계를 뽑을 수 있습니다.
- **앱 하나만** 재사용하고 매번 `open` → `close` 하는 것이 매번 `App()` 을 새로 만드는 것보다 빠릅니다.

## 응용

### 여러 폴더 → 한 폴더로 취합

```python
for src in [r"C:\2024", r"C:\2025", r"C:\2026"]:
    bulk_replace(src, replacements, output_dir=r"C:\all_updated")
```

### 정규표현식 필요할 때

HWP의 `replace_all` 은 정규식을 지원하지 않지만, 원문을 읽어와서 파이썬에서 처리 후 다시 쓸 수 있습니다.

```python
import re
from hwpapi import constants as const

app.open(path)
text = app.get_text(
    spos=const.ScanStartPosition.Document,
    epos=const.ScanEndPosition.Document,
)
new = re.sub(r"(\d{4})년 (\d{1,2})월", r"\1.\2", text)
# 전체 교체
app.actions.SelectAll.run()
app.api.Run("Delete")
app.insert_text(new)
app.save(path)
```

### 치환 실패 파일 별도 처리

```python
failed = [r for r in stats['files'] if r['replacements'] == 0]
for r in failed:
    print(f"치환된 부분 없음: {r['file']}")
```